### Initializations

In [54]:
import json
import pandas as pd
import requests

from pprint import pprint

base_url = "https://clinicaltrials.gov/api/v2/studies"
page_size = 10
request_timeout_seconds = 30

### First request

In [6]:
params = {
    "pageSize": page_size,
    "format": "json",
}

response = requests.get(
    base_url,
    params=params,
    timeout=request_timeout_seconds,
)

response.raise_for_status()

data = response.json()

### Inspect the general response

In [35]:
data.keys()

dict_keys(['studies', 'nextPageToken'])

In [37]:
len(data["studies"])

10

In [42]:
first_study = data["studies"][0]

type(first_study)

dict

In [43]:
first_study.keys()

dict_keys(['protocolSection', 'derivedSection', 'hasResults'])

In [47]:
print(json.dumps(first_study, indent=2))

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT00830635",
      "orgStudyIdInfo": {
        "id": "08-0498"
      },
      "secondaryIdInfos": [
        {
          "id": "AMCCRC-08-0498"
        }
      ],
      "organization": {
        "fullName": "University of Colorado, Denver",
        "class": "OTHER"
      },
      "briefTitle": "Multimedia Educational Program for Patients With Early-Stage Prostate Cancer or Breast Cancer",
      "officialTitle": "Cancer Information Service Research Consortium: Years 2006-2011 Program Narrative and Overview"
    },
    "statusModule": {
      "statusVerifiedDate": "2015-06",
      "overallStatus": "COMPLETED",
      "expandedAccessInfo": {
        "hasExpandedAccess": false
      },
      "startDateStruct": {
        "date": "2008-09"
      },
      "primaryCompletionDateStruct": {
        "date": "2013-04",
        "type": "ACTUAL"
      },
      "completionDateStruct": {
        "date": "2013-12",
        "type": "A

### Extract basic fields

In [48]:
protocol_section = first_study.get("protocolSection", {})

identification = protocol_section.get("identificationModule", {})
status = protocol_section.get("statusModule", {})
design = protocol_section.get("designModule", {})
conditions = protocol_section.get("conditionsModule", {})

In [50]:
identification

{'nctId': 'NCT00830635', 'orgStudyIdInfo': {'id': '08-0498'}, 'secondaryIdInfos': [{'id': 'AMCCRC-08-0498'}], 'organization': {'fullName': 'University of Colorado, Denver', 'class': 'OTHER'}, 'briefTitle': 'Multimedia Educational Program for Patients With Early-Stage Prostate Cancer or Breast Cancer', 'officialTitle': 'Cancer Information Service Research Consortium: Years 2006-2011 Program Narrative and Overview'}

In [51]:
study_summary = {
    "nct_id": identification.get("nctId"),
    "brief_title": identification.get("briefTitle"),
    "official_title": identification.get("officialTitle"),
    "overall_status": status.get("overallStatus"),
    "study_type": design.get("studyType"),
    "phases": design.get("phases"),
    "conditions": conditions.get("conditions"),
}

In [52]:
study_summary

{'nct_id': 'NCT00830635', 'brief_title': 'Multimedia Educational Program for Patients With Early-Stage Prostate Cancer or Breast Cancer', 'official_title': 'Cancer Information Service Research Consortium: Years 2006-2011 Program Narrative and Overview', 'overall_status': 'COMPLETED', 'study_type': 'INTERVENTIONAL', 'phases': ['NA'], 'conditions': ['Breast Cancer', 'Depression', 'Long-term Effects of Cancer Treatment', 'Prostate Cancer', 'Psychosocial Effects of Cancer and Its Treatment']}

In [53]:
def extract_study_summary(study: dict) -> dict:
    protocol_section = study.get("protocolSection", {})

    identification = protocol_section.get("identificationModule", {})
    status = protocol_section.get("statusModule", {})
    design = protocol_section.get("designModule", {})
    conditions = protocol_section.get("conditionsModule", {})

    return {
        "nct_id": identification.get("nctId"),
        "brief_title": identification.get("briefTitle"),
        "official_title": identification.get("officialTitle"),
        "overall_status": status.get("overallStatus"),
        "study_type": design.get("studyType"),
        "phases": design.get("phases"),
        "conditions": conditions.get("conditions"),
    }

In [57]:
df_studies = pd.DataFrame(
    extract_study_summary(study)
    for study in data["studies"]
)
df_studies

,nct_id,brief_title,official_title,overall_status,study_type,phases,conditions
0,NCT00830635,Multimedia Educational Program for Patients Wi...,Cancer Information Service Research Consortium...,COMPLETED,INTERVENTIONAL,[NA],"[Breast Cancer, Depression, Long-term Effects ..."
1,NCT06798935,Feasibility and Safety of Additional Injection...,Feasibility and Safety of Additional Injection...,COMPLETED,INTERVENTIONAL,[PHASE2],"[Rectovaginal Fistula, Anovaginal Fistula]"
2,NCT01990235,Improving Advance Care Planning by Preparing D...,Improving Advance Care Planning by Preparing D...,COMPLETED,INTERVENTIONAL,[NA],[Chronic Disease]
3,NCT01419535,Mifepristone Effects on Glucose Intolerance in...,"Effects of the Glucocorticoid Antagonist, Mife...",COMPLETED,INTERVENTIONAL,"[PHASE1, PHASE2]","[Endocrine Disease, Diabetes]"
4,NCT05944835,Validation of Numerical Simulation to Predict ...,Validation of Numerical Simulation to Predict ...,UNKNOWN,OBSERVATIONAL,None,[Surgery]
5,NCT05470335,Nurse-led Sheath Insertion in Cardiac Patients,"Patient Comfort, Success Rate, Procedure Time ...",RECRUITING,INTERVENTIONAL,[NA],[Coronary Artery Disease]
6,NCT03386435,The Effect of Remote Ischemic Preconditioning ...,The Effect of Remote Ischemic Preconditioning ...,COMPLETED,INTERVENTIONAL,[NA],"[Tissue Donors, Liver Transplantation, Ischemi..."
7,NCT05541835,Hematological Abnormalities in Children,Hematological Abnormalities In Children With C...,UNKNOWN,OBSERVATIONAL,None,[Hematologic Diseases]
8,NCT05665335,A Post-Market Study in Greece of a Minimally I...,A Post-Market Study of a Minimally Invasive Be...,COMPLETED,INTERVENTIONAL,[NA],[Breast Ptosis]
9,NCT02260635,A Study of Evacetrapib (LY2484595) in Japanese...,A Double-Blind Efficacy and Safety Study of Ev...,TERMINATED,INTERVENTIONAL,[PHASE3],[Hypercholesterolemia]


In [58]:
df_studies.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   nct_id          10 non-null     str   
 1   brief_title     10 non-null     str   
 2   official_title  10 non-null     str   
 3   overall_status  10 non-null     str   
 4   study_type      10 non-null     str   
 5   phases          8 non-null      object
 6   conditions      10 non-null     object
dtypes: object(2), str(5)
memory usage: 692.0+ bytes


In [60]:
df_study_conditions = (
    df_studies[["nct_id", "conditions"]]
    .explode("conditions")
    .rename(columns={"conditions": "condition_name"})
    .dropna()
    .reset_index(drop=True)
)
df_study_conditions

,nct_id,condition_name
0,NCT00830635,Breast Cancer
1,NCT00830635,Depression
2,NCT00830635,Long-term Effects of Cancer Treatment
3,NCT00830635,Prostate Cancer
4,NCT00830635,Psychosocial Effects of Cancer and Its Treatment
5,NCT06798935,Rectovaginal Fistula
6,NCT06798935,Anovaginal Fistula
7,NCT01990235,Chronic Disease
8,NCT01419535,Endocrine Disease
9,NCT01419535,Diabetes


### Data Quality checks

In [62]:
df_studies.isna().sum()

nct_id            0
brief_title       0
official_title    0
overall_status    0
study_type        0
phases            2
conditions        0
dtype: int64

In [64]:
df_studies["nct_id"].duplicated().sum()

np.int64(0)

In [65]:
df_studies["overall_status"].value_counts(dropna=False)

overall_status
COMPLETED     6
UNKNOWN       2
RECRUITING    1
TERMINATED    1
Name: count, dtype: int64

In [66]:
df_studies["study_type"].value_counts(dropna=False)

study_type
INTERVENTIONAL    8
OBSERVATIONAL     2
Name: count, dtype: int64